In [ ]:
# ==============================================================================
# DATA PREPARATION: MERGE LEAN BLS DATA
# Context: Merges pre-processed (lean) BLS labor data into GHGRP dataset.
# Input:
#   1. ghgrp_CT_data_final.csv
#   2. [Year].annual.singlefile_lean.csv (Files you just created)
# ==============================================================================

options(scipen = 999)
library(tidyverse)
library(readr)

# --- 1. Load Main GHGRP Data --------------------------------------------------
cat("\n[1/3] Loading Main GHGRP Data...\n")

main_file <- "ghgrp_CT_data_final.csv"

if(!file.exists(main_file)) {
  stop("❌ Error: 'ghgrp_CT_data_final.csv' not found. Please upload it to the Colab folder.")
}

df_main <- read_csv(main_file, show_col_types = FALSE) %>%
  mutate(
    NAICS3 = as.character(NAICS3),
    ghgrp_year = as.integer(ghgrp_year)
  )

cat(sprintf("   Loaded %d rows from main dataset.\n", nrow(df_main)))


# --- 2. Process LEAN BLS Files ------------------------------------------------
cat("\n[2/3] Processing Lean BLS Files...\n")

bls_data_list <- list()
years <- 2015:2023

process_bls_year <- function(yr) {

  # FIX: Look for 'lean.csv' instead of 'singlefile.csv'
  csv_pattern <- sprintf("%d.*lean\\.csv$", yr)
  files_found <- list.files(pattern = csv_pattern)

  if (length(files_found) == 0) {
    cat(sprintf("   ⚠️ No file found for %d. Skipping.\n", yr))
    return(NULL)
  }

  csv_name <- files_found[1]
  cat(sprintf("   Reading %d: %s ... ", yr, csv_name))

  tryCatch({
    # Since the file is already lean, we just read it.
    # We specify col_types to ensure codes are read as text (avoids type mismatches)
    clean_qcew <- read_csv(csv_name,
                           show_col_types = FALSE,
                           col_types = cols(
                             industry_code = col_character(),
                             year = col_integer(),
                             annual_avg_emplvl = col_double(),
                             avg_annual_pay = col_double()
                           )) %>%
      select(
        ghgrp_year = year,
        NAICS3 = industry_code,
        ind_total_jobs = annual_avg_emplvl,
        ind_avg_wage = avg_annual_pay
      )

    cat(sprintf("Done. (Found %d industries)\n", nrow(clean_qcew)))
    return(clean_qcew)

  }, error = function(e) {
    cat(sprintf("Failed. Error: %s\n", e$message))
    return(NULL)
  })
}

# Loop through years
for (y in years) {
  df_year <- process_bls_year(y)
  if (!is.null(df_year)) {
    bls_data_list[[as.character(y)]] <- df_year
  }
}

# Bind all years together
if(length(bls_data_list) > 0) {
  bls_master <- bind_rows(bls_data_list)
  cat(sprintf("   Compiled BLS Data: %d NAICS-Year combinations found.\n", nrow(bls_master)))
} else {
  stop("❌ No BLS data could be loaded. Please check your filenames.")
}


# --- 3. Merge and Export ------------------------------------------------------
cat("\n[3/3] Merging into Main Dataset...\n")

# Left Join: Keep all GHGRP rows, add BLS data where matches exist
df_final <- left_join(df_main, bls_master, by = c("ghgrp_year", "NAICS3"))

# Fill NA for HHI (Placeholder for next step)
if(!"ind_concentration_hhi" %in% names(df_final)) {
  df_final$ind_concentration_hhi <- NA
}

# Quick Stat Check
missing_matches <- sum(is.na(df_final$ind_avg_wage))
cat(sprintf("   Merge Complete. Rows with missing Labor Data: %d (%.1f%%)\n",
            missing_matches, (missing_matches/nrow(df_final))*100))

# Save
output_file <- "ghgrp_CT_data_final_with_BLS.csv"
write_csv(df_final, output_file)

cat(sprintf("✅ Success! New file saved as: %s\n", output_file))


[1/3] Loading Main GHGRP Data...
   Loaded 31145 rows from main dataset.

[2/3] Processing Lean BLS Files...
   Reading 2015: 2015.annual.singlefile_lean.csv ... Done. (Found 94 industries)
   Reading 2016: 2016.annual.singlefile_lean.csv ... Done. (Found 94 industries)
   Reading 2017: 2017.annual.singlefile_lean.csv ... Done. (Found 94 industries)
   Reading 2018: 2018.annual.singlefile_lean.csv ... Done. (Found 94 industries)
   Reading 2019: 2019.annual.singlefile_lean.csv ... Done. (Found 94 industries)
   Reading 2020: 2020.annual.singlefile_lean.csv ... Done. (Found 94 industries)
   Reading 2021: 2021.annual.singlefile_lean.csv ... Done. (Found 94 industries)
   Reading 2022: 2022.annual.singlefile_lean.csv ... Done. (Found 91 industries)
   Reading 2023: 2023.annual.singlefile_lean.csv ... Done. (Found 91 industries)
   Compiled BLS Data: 840 NAICS-Year combinations found.

[3/3] Merging into Main Dataset...
   Merge Complete. Rows with missing Labor Data: 269 (0.9%)
✅ Succes

In [3]:
# ==============================================================================
# DATA PREPARATION: MERGE & IMPUTE CENSUS CONCENTRATION DATA (FIXED)
# Context: Adds 'ind_concentration_hhi' to the dataset.
# Fixes:
#   1. Handles "One-Point" crash (when industry exists in 2022 but not 2017).
#   2. Parses complex ranges like "48-49 (106)".
#   3. Merges exact 3-digit matches first, then falls back to 2-digit parent.
# ==============================================================================

options(scipen = 999)
library(tidyverse)
library(readr)
library(stringr)

# --- 1. Load Target Data ------------------------------------------------------
cat("\n[1/5] Loading Target Data (from BLS Step)...\n")

target_file <- "ghgrp_CT_data_final_with_BLS.csv"

if(!file.exists(target_file)) {
  stop("❌ Error: 'ghgrp_CT_data_final_with_BLS.csv' not found.")
}

df_main <- read_csv(target_file, show_col_types = FALSE) %>%
  mutate(NAICS3 = as.character(NAICS3),
         ghgrp_year = as.integer(ghgrp_year))

# Remove placeholder if it exists
if("ind_concentration_hhi" %in% names(df_main)) {
  df_main <- df_main %>% select(-ind_concentration_hhi)
}

cat(sprintf("   Loaded %d rows.\n", nrow(df_main)))


# --- 2. Process Census Files (2017 & 2022) ------------------------------------
cat("\n[2/5] Processing Census Concentration Files...\n")

process_census_file <- function(year) {
  # Regex to match both "ECNSIZE..." and "concentration_..."
  pattern <- sprintf("(concentration|ECNSIZE).*?%d.*\\.csv", year)
  files <- list.files(pattern = pattern, ignore.case = TRUE)

  if(length(files) == 0) stop(sprintf("❌ No file found for year %d", year))

  filename <- files[1]
  cat(sprintf("   Reading %d data from: %s ... ", year, filename))

  # Read all cols as character
  raw <- read_csv(filename, col_types = cols(.default = "c"), show_col_types = FALSE)

  # Dynamic Column Finding
  col_naics <- names(raw)[str_detect(names(raw), "(?i)NAICS.*code")]
  col_hhi   <- names(raw)[str_detect(names(raw), "(?i)HHI")]

  if(length(col_naics) == 0 || length(col_hhi) == 0) stop("❌ Columns not found.")

  # Extract and Clean
  clean <- raw %>%
    select(NAICS_Code = all_of(col_naics[1]),
           HHI_Raw = all_of(col_hhi[1])) %>%
    filter(!is.na(NAICS_Code)) %>%
    mutate(
      Year = year,
      # Convert HHI to numeric (Forces "D" to NA)
      HHI = suppressWarnings(as.numeric(HHI_Raw))
    ) %>%
    filter(!is.na(HHI))

  # --- RANGE EXPANSION (Fixed for "48-49 (106)") ---
  ranges <- clean %>% filter(str_detect(NAICS_Code, "-"))
  clean_no_ranges <- clean %>% filter(!str_detect(NAICS_Code, "-"))

  expanded_list <- list()
  if(nrow(ranges) > 0) {
    for(i in 1:nrow(ranges)) {
      code_str <- ranges$NAICS_Code[i]
      hhi_val  <- ranges$HHI[i]

      parts <- str_split(code_str, "-")[[1]]
      # Check if valid range format "XX-YY..."
      if(length(parts) == 2) {
        # Clean the start/end numbers (remove parenthesis/text)
        start_num <- as.integer(str_extract(parts[1], "^[0-9]+"))
        end_num   <- as.integer(str_extract(parts[2], "^[0-9]+"))

        if(!is.na(start_num) && !is.na(end_num)) {
          seq_codes <- seq(start_num, end_num)
          expanded_list[[i]] <- data.frame(
            NAICS_Code = as.character(seq_codes),
            HHI = hhi_val,
            Year = year
          )
        }
      }
    }
  }

  if(length(expanded_list) > 0) {
    clean <- bind_rows(clean_no_ranges, bind_rows(expanded_list))
  } else {
    clean <- clean_no_ranges
  }

  # Final Cleanup: Remove non-digits from codes
  clean <- clean %>% mutate(NAICS_Code = str_extract(NAICS_Code, "^[0-9]+"))

  cat(sprintf("Done (%d valid rows).\n", nrow(clean)))
  return(clean)
}

df_2017 <- process_census_file(2017)
df_2022 <- process_census_file(2022)
census_combined <- bind_rows(df_2017, df_2022)


# --- 3. Imputation (The Fix) --------------------------------------------------
cat("\n[3/5] Imputing Missing Years (2015-2023)...\n")

unique_naics <- unique(census_combined$NAICS_Code)
years_seq <- 2015:2023
imputed_list <- list()

for(code in unique_naics) {
  sub <- census_combined %>% filter(NAICS_Code == code)

  if(nrow(sub) > 0) {
    # --- LOGIC FIX: Handle 1-point case vs 2-point case ---
    if(nrow(sub) >= 2) {
      # We have 2017 AND 2022 -> Interpolate
      interp <- approx(x = sub$Year, y = sub$HHI, xout = years_seq, method = "linear", rule = 2)
      y_vals <- interp$y
    } else {
      # We ONLY have 1 year (likely 2022) -> Constant Extrapolation
      # Assumption: The 2022 structure is the best proxy for 2015-2023
      y_vals <- rep(sub$HHI[1], length(years_seq))
    }

    imputed_list[[code]] <- data.frame(
      NAICS_Code = code,
      ghgrp_year = years_seq,
      ind_concentration_hhi = y_vals
    )
  }
}

hhi_master <- bind_rows(imputed_list)
cat(sprintf("   Imputation Grid Created for %d industries.\n", n_distinct(hhi_master$NAICS_Code)))


# --- 4. Merge -----------------------------------------------------------------
cat("\n[4/5] Merging into Main Data...\n")

# Join 1: Exact 3-Digit Match
df_merged <- df_main %>%
  left_join(hhi_master, by = c("NAICS3" = "NAICS_Code", "ghgrp_year")) %>%
  rename(hhi_exact = ind_concentration_hhi)

# Join 2: Parent 2-Digit Match (Fallback)
df_merged <- df_merged %>%
  mutate(NAICS2 = substr(NAICS3, 1, 2)) %>%
  left_join(hhi_master, by = c("NAICS2" = "NAICS_Code", "ghgrp_year")) %>%
  rename(hhi_parent = ind_concentration_hhi) %>%
  mutate(
    ind_concentration_hhi = coalesce(hhi_exact, hhi_parent),
    match_type = case_when(
      !is.na(hhi_exact) ~ "Exact (3-Digit)",
      !is.na(hhi_parent) ~ "Parent (2-Digit)",
      TRUE ~ "Missing"
    )
  ) %>%
  select(-hhi_exact, -hhi_parent, -NAICS2)


# --- 5. Export ----------------------------------------------------------------
cat("\n[5/5] Final Checks...\n")

coverage <- df_merged %>% count(match_type) %>% mutate(pct = n/sum(n)*100)
print(coverage)

output_file <- "ghgrp_CT_data_final_with_BLS_CS.csv"
write_csv(df_merged, output_file)

cat(sprintf("\n✅ Success! Saved: %s\n", output_file))


[1/5] Loading Target Data (from BLS Step)...
   Loaded 31145 rows.

[2/5] Processing Census Concentration Files...
   Reading 2017 data from: concentration_2017.csv ... Done (19 valid rows).
   Reading 2022 data from: concentration_2022.csv ... Done (1312 valid rows).

[3/5] Imputing Missing Years (2015-2023)...
   Imputation Grid Created for 1312 industries.

[4/5] Merging into Main Data...

[5/5] Final Checks...
# A tibble: 3 × 3
  match_type           n     pct
  <chr>            <int>   <dbl>
1 Exact (3-Digit)  30806 98.9   
2 Missing            317  1.02  
3 Parent (2-Digit)    22  0.0706

✅ Success! Saved: ghgrp_CT_data_final_with_BLS_CS.csv


In [5]:
# ============================================================
# HLM for GHGRP discrepancies — WITH 2ND STAGE MECHANISMS (FINAL)
# Paper: "Industrial architecture, not corporate governance..."
# Dataset: ghgrp_CT_data_final_with_BLS_CS.csv (Includes Wages, Jobs, HHI)
# ============================================================

# ----- 0. AUTO-INSTALL DEPENDENCIES -----
options(scipen = 999)
options(repos = c(CRAN = "https://cloud.r-project.org"))

required_packages <- c("tidyverse", "lme4", "lmerTest", "splines", "broom.mixed", "MuMIn", "stargazer")
new_packages <- required_packages[!(required_packages %in% installed.packages()[,"Package"])]
if(length(new_packages)) {
  cat("Installing missing packages...\n")
  install.packages(new_packages)
}

suppressPackageStartupMessages({
  library(tidyverse)
  library(lme4)
  library(lmerTest)
  library(splines)
  library(broom.mixed)
  library(MuMIn)
  library(stargazer)
})

# ============================================================
# 1. DATA LOADING & PREP
# ============================================================
cat("\n[1/5] Loading Data...\n")

# !!! UPDATED: Reading the file with Census (HHI) + BLS data !!!
input_file <- "ghgrp_CT_data_final_with_BLS_CS.csv"

if(!file.exists(input_file)) stop("❌ File not found. Please run the Census merge script first!")
df <- read_csv(input_file, show_col_types = FALSE)

# Check for HHI Data Availability
has_hhi <- !all(is.na(df$ind_concentration_hhi))
if(has_hhi) {
  cat("✅ HHI Data Detected. Full model will run.\n")
} else {
  cat("⚠️ Warning: HHI column is empty. Running partial model.\n")
}

# Basic Cleaning & Transformations
df <- df %>%
  mutate(
    ghgrp_emissions_tons = ifelse(ghgrp_emissions_tons < 1, 1, ghgrp_emissions_tons),
    rel_error = abs(emissions_discrepancy) / ghgrp_emissions_tons,
    rel_error_win = ifelse(rel_error > quantile(rel_error, 0.99, na.rm=T),
                           quantile(rel_error, 0.99, na.rm=T), rel_error),
    log_rel_error = log1p(rel_error_win),
    log_size = log(ghgrp_emissions_tons),
    log_size_scaled = scale(log_size),

    # Scale Predictors (Standardized Z-scores)
    ind_wage_z = as.numeric(scale(ind_avg_wage)),
    ind_jobs_z = as.numeric(scale(ind_total_jobs))
)

# Safe Scale HHI
if(has_hhi) {
  df$ind_hhi_z <- as.numeric(scale(df$ind_concentration_hhi))
} else {
  df$ind_hhi_z <- 0
}

cat(sprintf("Loaded %d rows. Unique NAICS3 Sectors: %d\n", nrow(df), n_distinct(df$NAICS3)))

# ============================================================
# 2. MAIN HLM MODEL (First Stage)
# ============================================================
cat("\n[2/5] Running Main HLM (Variance Decomposition)...\n")

formula_main <- log_rel_error ~ ns(log_size_scaled, df=3) + as.factor(ghgrp_year) +
                (1 | ghgrp_id) +
                (1 | ghgrp_county_name) +
                (1 | NAICS3) +
                (1 | ghgrp_parent_companies)

model_main <- lmer(formula_main, data = df, control = lmerControl(optimizer = "bobyqa"))

print(as.data.frame(VarCorr(model_main)) %>% select(grp, vcov))

# ============================================================
# 3. EXTRACT INDUSTRY EFFECTS (BLUPs)
# ============================================================
cat("\n[3/5] Extracting Industry Random Effects...\n")

re_extract <- ranef(model_main, condVar = TRUE)
ind_effects <- as.data.frame(re_extract$NAICS3) %>%
  rownames_to_column("NAICS3") %>%
  rename(Industry_Bias = `(Intercept)`)

# ============================================================
# 4. PREPARE SECOND STAGE DATA
# ============================================================
cat("\n[4/5] Aggregating Mechanism Variables...\n")

ind_features <- df %>%
  group_by(NAICS3) %>%
  summarise(
    Mean_Avg_Wage   = mean(ind_avg_wage, na.rm=TRUE),
    Mean_Total_Jobs = mean(ind_total_jobs, na.rm=TRUE),
    Mean_HHI        = if(has_hhi) mean(ind_concentration_hhi, na.rm=TRUE) else 0,
    Mean_ROA        = mean(ind_ROA_z, na.rm=TRUE),
    Mean_Leverage   = mean(ind_Leverage_z, na.rm=TRUE)
  ) %>%
  mutate(NAICS3 = as.character(NAICS3))

stage2_data <- left_join(ind_effects, ind_features, by = "NAICS3")

stage2_data_scaled <- stage2_data %>%
  mutate(
    Wage_Z = scale(Mean_Avg_Wage),
    Jobs_Z = scale(Mean_Total_Jobs),
    HHI_Z  = if(has_hhi) scale(Mean_HHI) else 0,
    ROA_Z  = scale(Mean_ROA),
    Lev_Z  = scale(Mean_Leverage)
  )

# ============================================================
# 5. SECOND STAGE REGRESSION
# ============================================================
cat("\n[5/5] Running Second Stage Mechanism Analysis...\n")

if(has_hhi) {
  # FULL MODEL (With Census Data)
  mod_mech_1 <- lm(Industry_Bias ~ HHI_Z + Wage_Z + Jobs_Z, data = stage2_data_scaled)
  mod_mech_2 <- lm(Industry_Bias ~ HHI_Z + Wage_Z + Jobs_Z + ROA_Z + Lev_Z, data = stage2_data_scaled)

  stargazer(mod_mech_1, mod_mech_2, type = "text",
            title = "Drivers of Industry-Level Discrepancy (Full Model)",
            dep.var.labels = "Industry Random Effect (Bias)",
            covariate.labels = c("Concentration (HHI)", "Labor Sophistication (Wage)",
                                 "Industry Scale (Jobs)", "Profitability", "Leverage"),
            omit.stat = c("f", "ser"))
} else {
  # PARTIAL MODEL (Backup)
  mod_mech_1 <- lm(Industry_Bias ~ Wage_Z + Jobs_Z, data = stage2_data_scaled)
  mod_mech_2 <- lm(Industry_Bias ~ Wage_Z + Jobs_Z + ROA_Z + Lev_Z, data = stage2_data_scaled)

  stargazer(mod_mech_1, mod_mech_2, type = "text",
            title = "Drivers of Industry-Level Discrepancy (No HHI)",
            dep.var.labels = "Industry Random Effect (Bias)",
            omit.stat = c("f", "ser"))
}

cat("\n--- INTERPRETATION ---\n")
cat("1. HHI_Z (Concentration): Positive = Monopolies have higher error.\n")
cat("2. Wage_Z (Complexity): Positive = High-tech/complex sectors have higher error.\n")

Installing missing packages...


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘nloptr’, ‘RcppEigen’, ‘numDeriv’, ‘coda’





[1/5] Loading Data...
✅ HHI Data Detected. Full model will run.
Loaded 31145 rows. Unique NAICS3 Sectors: 52

[2/5] Running Main HLM (Variance Decomposition)...
                     grp       vcov
1               ghgrp_id 0.34462295
2 ghgrp_parent_companies 0.04913047
3      ghgrp_county_name 0.02301040
4                 NAICS3 0.76568727
5               Residual 0.11605714

[3/5] Extracting Industry Random Effects...

[4/5] Aggregating Mechanism Variables...

[5/5] Running Second Stage Mechanism Analysis...

Drivers of Industry-Level Discrepancy (Full Model)
                                 Dependent variable:      
                            ------------------------------
                            Industry Random Effect (Bias) 
                                  (1)            (2)      
----------------------------------------------------------
Concentration (HHI)             -0.085          -0.096    
                                (0.157)        (0.161)    
                    